# 1 — Reformat: CFD Brackish → a COCO **keypoints** data root

**Question this branch answers.** `crop-counter` is a frozen-DINOv3 point-heatmap *counter*. A parked branch bolted a **box** head onto the same decoder and trained it on the Brackish source of the Community Fish Detection Dataset (CFD); rescored as points (point-in-box F1 over the identical val frames, [`docs/free_first_step.json`](docs/free_first_step.json)) it reached **F1 0.758** (best) / 0.746 (last) against released **RF-DETR-Nano at 0.782**. This branch runs the package's **native point task** on those same frames. Does dropping the `wh`/`off` geometry branch cost the heat branch anything, or free capacity for it? One seed, so the answer is a *read*, not a claim.

**Layout — nothing touches the laptop.**

| Location | Holds |
|---|---|
| `/content/` (VM SSD, ephemeral) | CFD metadata, the Brackish frames (re-fetched each session — cheap on the LILA/GCS pipe), the keypoints root |
| Google Drive `frozen-trunk-detection/` | converted DINOv3 backbone weights (`weights/`), the run dir (`runs/brackish_points_s0/`), `results/points/` (tables, calibration, figures) |

**Do not put images on Drive** — per-file Drive API latency makes 14.7k JPEG reads crawl.

**The frames are published at 960×540**, so `--max-side 1024` resizes **nothing** here: every `cfd_scale` comes back 1.0 and the geometry is untouched. The flag is kept because it is what the box run used and what the other CFD sources need — not because it does anything to Brackish.

---

**This notebook (1 of 4)** turns the CFD master metadata into a per-source manifest, subsets Brackish on the published `is_train` split, fetches its frames to the VM disk, converts box centres to COCO keypoints in a **separate** data root, verifies that conversion round-trips, and writes Table 1.

The four notebooks: **1_reformat** → `2_training` (the run, τ calibration, Table 2) → `3_inference` → `4_evaluate`.

## 1 · Drive, paths, code

Mounts Drive, fixes the paths every cell below uses, clones this branch and installs it editable. Safe to re-run: the clone is wiped and redone each time.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, time, shutil, importlib, pathlib
from pathlib import Path

from IPython.display import Image as IPImage, display

DRIVE  = '/content/drive/MyDrive/frozen-trunk-detection'
REPO   = '/content/crop-counter'
DATA   = '/content/data/brackish'          # the COCO *bbox* subset — the pixels live here
POINTS = '/content/data/brackish_points'   # the COCO *keypoints* root the trainer reads
CFD    = '/content/cfd'
RUN    = f'{DRIVE}/runs/brackish_points_s0'
RES    = f'{DRIVE}/results/points'
for d in (CFD, f'{DRIVE}/weights', f'{DRIVE}/runs', f'{DRIVE}/results',
          f'{DRIVE}/results/manifest', RES, f'{RES}/figures'):
    os.makedirs(d, exist_ok=True)
print(os.listdir(DRIVE))

%cd /content
!rm -rf crop-counter
!git clone --branch poc/fish-points --depth 1 https://github.com/InsightML/crop-counter.git
%cd /content/crop-counter
!git log --oneline -3
# cfd = ijson (streams the 1.9M-record CFD metadata); dev = nbconvert/ruff/pytest;
# portal = the hosted-API client. No detection extra on this branch: no boxes, no COCO AP.
!pip install -q -e ".[dev,portal,cfd]"

# A running kernel never re-reads site-packages' .pth files, so the editable install is
# invisible to THIS process until a restart (subprocess `!python -m ...` calls see it fine).
for p in ('/content/crop-counter/src', '/content/crop-counter/examples/FishDetection/scripts'):
    if p not in sys.path:
        sys.path.insert(0, p)
importlib.invalidate_caches()
import cropcounter, torch
import nb_helpers as nbh
print('cropcounter', cropcounter.__file__, '| torch', torch.__version__,
      '| cuda', torch.cuda.is_available())

# Backbone: Meta's DINOv3 checkpoint is gated; the file on Drive is the ungated timm
# re-host converted to Meta's parameter names. Symlink, never copy — it is 350 MB and the
# clone is thrown away every session anyway.
os.makedirs('weights', exist_ok=True)
BACKBONE = 'dinov3_convnext_base_pretrain_lvd1689m-801f2ba9.pth'
if not os.path.lexists(f'weights/{BACKBONE}'):
    os.symlink(f'{DRIVE}/weights/{BACKBONE}', f'weights/{BACKBONE}')
# decoder_best.pt (the shipped wheat decoder, used only by 2_training's backbone gate)
# is tracked in git; a Drive copy, if one exists, wins.
if os.path.exists(f'{DRIVE}/weights/decoder_best.pt'):
    if os.path.lexists('weights/decoder_best.pt'):
        os.remove('weights/decoder_best.pt')
    os.symlink(f'{DRIVE}/weights/decoder_best.pt', 'weights/decoder_best.pt')
    print('decoder_best.pt <- Drive')
else:
    print("decoder_best.pt: using the repo's own (tracked in git)")
!ls -l weights/

## 1b · What is already on this VM

Everything below is guarded on these three flags, so the notebook is safe to re-run on a
live runtime: it rebuilds only what is missing. Set `REBUILD = True` to force a clean
rebuild from the metadata down.

In [ ]:
REBUILD = False

def _n_files(path):
    return len(os.listdir(path)) if os.path.isdir(path) else 0

META = f'{CFD}/community_fish_detection_dataset.json.zip'
HAVE_MANIFEST = os.path.exists(f'{DRIVE}/results/manifest/manifest.csv') and not REBUILD
HAVE_DATA = (os.path.exists(f'{DATA}/val/annotations.json')
             and _n_files(f'{DATA}/val/images') > 0 and not REBUILD)
HAVE_POINTS = os.path.exists(f'{POINTS}/val/annotations.json') and not REBUILD
print(f'manifest {HAVE_MANIFEST} | bbox subset {HAVE_DATA} '
      f'(train {_n_files(f"{DATA}/train/images")}, val {_n_files(f"{DATA}/val/images")} frames) '
      f'| points root {HAVE_POINTS}')

## 2 · CFD metadata → per-source manifest

47 MB zipped; the JSON inside is *streamed* (`ijson`), never `json.load`-ed. The manifest is the evidence behind every subsetting decision: empty-image fraction, `is_train` balance, box-size percentiles, stride-4 centre-cell collision rate, licence. Brackish was chosen for this head shape because its **stride-4 centre-cell collision rate is 0.0 %** — no two fish in a frame want the same output cell, so a one-peak-per-cell head can represent every animal. It lands on Drive so notebooks 3 and 4 never recompute it.

In [ ]:
if not HAVE_MANIFEST:
    if not os.path.exists(META):
        !wget -q --show-progress -O {META} https://lilawildlife.blob.core.windows.net/lila-wildlife/community-fish-detection-dataset/community_fish_detection_dataset.json.zip
    t0 = time.time()
    !python -m cropcounter.cfd manifest --metadata {META} --out {DRIVE}/results/manifest --no-progress
    print(f'manifest in {time.time() - t0:.0f}s')
else:
    print('manifest already on Drive — skipping')

import csv
manifest_rows = list(csv.DictReader(open(f'{DRIVE}/results/manifest/manifest.csv')))
name_col = next(c for c in manifest_rows[0].keys() if c.lower() in ('dataset', 'source', 'name'))
brackish_row = next(r for r in manifest_rows if 'rackish' in r[name_col])
for key in ('source', 'licence', 'permissive', 'n_images', 'n_boxes', 'n_empty_images',
            'empty_fraction', 'boxes_per_image', 'is_train_true', 'is_train_false',
            'sequences', 'median_width', 'median_height', 'box_px_p50', 'box_rel_p50',
            'collision_rate'):
    if key in brackish_row:
        print(f'{key:>18}: {brackish_row[key]}')

## 3 · Subset — **all** of Brackish, on the published `is_train` split

`--train-cap 100000 --val-cap 100000` is "take everything": Brackish has 14,674 frames, so both caps are inert and the subset is the whole source. That is deliberate. CFD publishes an `is_train` flag per frame and the box run honoured it; **re-splitting would break frame-for-frame comparability with the box run and risk sequence leakage** (Brackish is 89 fixed-camera sequences — a random split would put neighbouring frames of the same fish on both sides). No re-split, ever.

The source string is resolved out of `manifest.csv` rather than hard-coded, then asserted — if CFD ever renames the source, this fails loudly instead of silently subsetting nothing.

In [ ]:
BRACKISH_SOURCE = brackish_row[name_col]
print('Brackish source string:', repr(BRACKISH_SOURCE))
assert BRACKISH_SOURCE == 'brackish_dataset', (
    f'CFD renamed the Brackish source to {BRACKISH_SOURCE!r} — every downstream '
    'ensure-data cell hard-codes "brackish_dataset" and must be updated')

if not HAVE_DATA:
    if not os.path.exists(META):
        !wget -q --show-progress -O {META} https://lilawildlife.blob.core.windows.net/lila-wildlife/community-fish-detection-dataset/community_fish_detection_dataset.json.zip
    !rm -rf {DATA}
    t0 = time.time()
    !python -m cropcounter.cfd subset --metadata {META} --out {DATA} --sources "{BRACKISH_SOURCE}" --train-cap 100000 --val-cap 100000 --seed 0 --no-progress
    print(f'subset in {time.time() - t0:.0f}s')
else:
    print('bbox subset already on this VM — skipping')
print(json.dumps(json.load(open(f'{DATA}/subset_summary.json')), indent=1)[:1500])
!wc -l < {DATA}/download_list.txt

## 4 · Fetch the frames to the VM disk

Resize on write, not at train time — except that here there is nothing to resize: Brackish is published at **960×540**, `--max-side 1024` never upscales, so every `cfd_scale` is 1.0 and `annotations.json` is identical to `annotations.native.json`. The flag stays for parity with the box run (and because the other CFD sources genuinely need it). The `gcs` mirror is the faster of the two from Colab.

In [ ]:
if not HAVE_DATA:
    t0 = time.time()
    !python -m cropcounter.cfd fetch --subset {DATA} --max-side 1024 --workers 32 --mirror gcs --no-progress
    print(f'fetched in {time.time() - t0:.0f}s')
else:
    print('frames already on this VM — skipping')
!du -sh {DATA}/train/images {DATA}/val/images
print('train frames:', _n_files(f'{DATA}/train/images'), '| val frames:', _n_files(f'{DATA}/val/images'))

scales = {im['cfd_scale'] for im in json.load(open(f'{DATA}/val/annotations.json'))['images']}
print('distinct cfd_scale values in val:', scales, '-> 1.0 means the 960x540 frames were untouched')

## 5 · `cfd points` — box centres → COCO **keypoints**, in a separate root

Not optional and not cosmetic. Three things happen here, and each of them is a silent failure if skipped:

1. **Every box becomes one keypoint** at its centre (`cx = x + w/2`, `cy = y + h/2`, visibility 2). A bbox-only COCO document has **no `keypoints` key at all**, so `parse_coco_keypoints` would build one record per image, attach nothing, and train happily on an empty dataset with a finite, falling loss.
2. **Ids are renumbered** to consecutive 1-based ints in file order. `parse_coco_keypoints` does `int(image["id"])` and CFD ids are *strings* (`"brackish_dataset_2019-02-21_…jpg"`), so the raw document raises `ValueError` on load. The original id survives on every record as `cfd_image_id`, and `cfd_id_map.json` maps original → new int — **that map is the join key back to the box run's string-keyed `predictions.json`**, which notebook 4 needs to compare the two systems frame for frame.
3. **A separate output root**, because `crop_dataset.resolve_annotations` picks `annotations.json` first: a points file written beside the bbox one could never be the file the loader opens. `{POINTS}/<split>/images` is an absolute **symlink** back to the subset, so the pixels are not duplicated.

Empty frames are **retained** — 60 % of Brackish frames contain no fish, and those real negatives are exactly what a counter needs and what a crop dataset almost never has.

In [ ]:
if not HAVE_POINTS:
    !rm -rf {POINTS}
    !python -m cropcounter.cfd points --subset {DATA} --out {POINTS}
else:
    print('points root already on this VM — skipping')

points_summary = json.load(open(f'{POINTS}/points_summary.json'))
print(json.dumps(points_summary, indent=2))
!ls -l {POINTS}/train {POINTS}/val

id_map = json.load(open(f'{POINTS}/cfd_id_map.json'))
example = list(id_map['val'].items())[:2]
print(f"cfd_id_map.json: {len(id_map['val']):,} val entries, e.g. {example} "
      "— original CFD string id -> new int id (the join back to the box run's predictions)")

## 6 · Round-trip verify — the conversion, through the loader the trainer uses

The only check that matters is the one the *training path* performs, so this loads the written file with `parse_coco_keypoints(..., labels=["fish"])` — the exact call `load_splits` makes — and asserts it against the bbox document it came from:

- one record per image in the bbox document (no frame lost),
- as many points as there were boxes (no annotation lost, none invented),
- as many empty records as `points_summary.json` counted (**the negatives survived**).

It then asserts the **raw** bbox document still raises `ValueError` through the same function. That is finding C from the parked branch, pinned live: the string-id failure is the *loud* half of the trap, and it must stay loud — the silent half (a bbox document with int ids loading as an empty dataset) is what the separate root and this assertion exist to make impossible.

Val is asserted to the exact published numbers because notebook 4 compares against the box run on **these** 3,127 frames; train is printed.

In [ ]:
from cropcounter.crop_dataset import parse_coco_keypoints, resolve_annotations

EXPECTED_VAL = {'images': 3127, 'points': 1965, 'empty': 1740}
observed = {}
for split in ('train', 'val'):
    bbox_doc = json.load(open(f'{DATA}/{split}/annotations.json'))
    n_images_bbox = len(bbox_doc['images'])
    n_boxes = sum(1 for a in bbox_doc['annotations'] if a.get('bbox'))

    recs = parse_coco_keypoints(resolve_annotations(pathlib.Path(f'{POINTS}/{split}'), fmt='coco'),
                                labels=['fish'])
    n_points = sum(len(r.points) for r in recs)
    n_empty = sum(1 for r in recs if not r.points)

    assert len(recs) == n_images_bbox, f'{split}: {len(recs)} records vs {n_images_bbox} images'
    assert n_points == n_boxes, f'{split}: {n_points} points vs {n_boxes} boxes'
    assert n_empty == points_summary[split]['n_empty'], f'{split}: empty-frame count drifted'
    assert n_points == points_summary[split]['n_points']

    # Finding C, pinned live: the RAW bbox document must still be unloadable.
    try:
        parse_coco_keypoints(pathlib.Path(f'{DATA}/{split}/annotations.json'), labels=['fish'])
        raise AssertionError(
            f'{split}: the raw bbox document loaded without raising — CFD ids are no longer '
            'strings, so the SILENT half of the trap (bbox doc -> empty dataset) is now live')
    except ValueError as exc:
        assert 'invalid literal for int()' in str(exc), str(exc)

    observed[split] = {'images': len(recs), 'points': n_points, 'empty': n_empty}
    print(f'{split:>5}: {len(recs):,} records | {n_points:,} points | {n_empty:,} empty '
          f'({100 * n_empty / len(recs):.1f}%) | raw bbox doc raises ValueError ')

assert observed['val'] == EXPECTED_VAL, (
    f"val drifted from the published Brackish split: {observed['val']} vs {EXPECTED_VAL} — "
    'the box run scored those exact frames, so notebook 4 would no longer be comparable')
print('\nval matches the published split exactly:', EXPECTED_VAL)
print('train (printed, not asserted — the box run scored val):', observed['train'])

## 7 · Table 1

Printed from the artefacts (`points_summary.json`), not recomputed: if the summary and the annotation files ever disagreed, § 6 above is what should fail, not this table quietly papering over it. Written to Drive as `results/points/table1.md` + `.json` for the report.

In [ ]:
table1_rows = nbh.table1_rows(POINTS)
table1_md = nbh.render_table1(table1_rows)
print(table1_md)

with open(f'{RES}/table1.md', 'w') as fh:
    fh.write('# Table 1 — CFD Brackish as a point-task data root\n\n')
    fh.write(table1_md)
    fh.write('\nBox centres from the CFD `brackish_dataset` source, published `is_train` '
             'split, frames at 960x540 (unresized). Empty frames retained as negatives.\n')
with open(f'{RES}/table1.json', 'w') as fh:
    json.dump({'rows': table1_rows, 'source': BRACKISH_SOURCE,
               'points_summary': points_summary}, fh, indent=2)
print('wrote', f'{RES}/table1.md', 'and', f'{RES}/table1.json')

## 8 · What the data looks like

A seeded sample per split — two thirds drawn from frames that contain fish, the rest empty, because a uniform sample of a 60 %-empty dataset is mostly water — with the ground-truth **points** as green dots (≈10 px radius, in frame pixels). Then the points-per-frame distribution, which is the shape the counter is actually scored on: a large zero spike, a long thin tail, and a mean well under 1.

These are the point-task twins of the box run's `data_samples_*` / `data_stats` figures. Saved to Drive and reused in the report.

In [ ]:
for split in ('train', 'val'):
    path = nbh.plot_point_samples(POINTS, split, f'{RES}/figures/data_samples_{split}.png',
                                  n=12, seed=0)
    display(IPImage(str(path)))

path = nbh.plot_count_distribution(POINTS, f'{RES}/figures/count_distribution.png')
display(IPImage(str(path)))
!ls -l {RES}/figures

## 9 · Next

`2_training.ipynb` — the backbone metric-reproduction gate, the labels/augment guards, the 8-epoch run, τ calibration and Table 2. Keep this VM alive if you can: the frames and the keypoints root just built are on its disk, and its § 1b rebuilds them in ~5 minutes if not.